# Tutorial 05 - Exercise 02: Titanic Predictive Pipeline & Evaluation
Builds an end-to-end pipeline: cleaning, feature engineering, categorical encoding, train/test split, feature scaling, and classifier evaluation.


In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

raw_df = sns.load_dataset('titanic')
print("Columns loaded:", list(raw_df.columns))


### 1. Data Cleaning & Feature Engineering


In [ ]:
# Drop redundant or heavily missing columns
df = raw_df.drop(columns=['deck', 'alive', 'embark_town', 'who', 'class'])

# Impute missing values
df['age'] = SimpleImputer(strategy='median').fit_transform(df[['age']])
df['embarked'] = SimpleImputer(strategy='most_frequent').fit_transform(df[['embarked']]).ravel()

# Create engineered features
df['FamilySize'] = df['sibsp'] + df['parch'] + 1
df['IsAlone'] = np.where(df['FamilySize'] == 1, 1, 0)
df.drop(columns=['sibsp', 'parch'], inplace=True)

# Encode categorical variables
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True, dtype=int)
for b in ['adult_male', 'alone']:
    df[b] = df[b].astype(int)

display(df.head())


### 2. Train-Test Split & Scaling


In [ ]:
X = df.drop(columns=['survived'])
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scale_cols = ['age', 'fare', 'FamilySize']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])


### 3. Model Training & Evaluation


In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print("Logistic Regression Report:\n", classification_report(y_test, y_pred_lr))

print(f"\nKNN Accuracy:                 {accuracy_score(y_test, y_pred_knn):.4f}")
print("KNN Report:\n", classification_report(y_test, y_pred_knn))
